In [ ]:
# RSNA Knee — submission (offline)
# Code and weights come from the private dataset built by scripts/package_kaggle.py.
import glob, os, subprocess, sys, time
T0 = time.time()
# mount points differ between Kaggle versions: look a few levels down, never recursively (820k files)
def find(pattern):
    return next(p for depth in range(4) for p in glob.glob('/kaggle/input/' + '*/' * depth + pattern))
KNEE = os.path.dirname(os.path.dirname(find('knee/infer.py')))
DATA = os.path.dirname(find('test_series.csv'))
print(KNEE, DATA)
# DICOM decoders (JPEG lossless / JPEG 2000) are not in the Kaggle image
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', f'{KNEE}/wheels',
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'], check=False)
sys.path.insert(0, KNEE)

In [ ]:
from knee.infer import run
# which trained runs of the dataset to blend (glob patterns on their folder names)
MODELS = ['*e03*']
models = sorted({m for pat in MODELS for m in glob.glob(f'{KNEE}/models/{pat}')})
print(models)
sub = run(DATA, models, out_csv='submission.csv', cache_dir='/tmp/knee_cache', workers=4)
print(sub.shape, f'{(time.time() - T0) / 60:.1f} min')